# Projet 12 — ONCFM : Notebook 2 — Application de détection de faux billets

**Auteur : Roger AGNEROH**  
**Objectif :** appliquer le modèle de régression logistique déjà entraîné à un nouveau fichier de billets, sans réentraîner le modèle.

Ce notebook est **indépendant du notebook d’analyse et d’entraînement**. Il ne dépend ni de `X_train` ni de variables créées dans le premier notebook.

**Fichiers à placer dans le même dossier que ce notebook :** `modele_ONCFM.pkl`, `scaler_ONCFM.pkl`, `mediane_ONCFM.pkl` et `billets_production.csv`.

## 13. Préparation de l’application

L’application charge le modèle et les transformations enregistrées, contrôle les six mesures géométriques, puis prédit la classe de chaque billet.

### 13.1 Importation des bibliothèques et configuration

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

DOSSIER = Path.cwd()  # Ouvrir Jupyter depuis le dossier du Projet 12
VARIABLES = ["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"]
print("Variables attendues :", VARIABLES)
print("Dossier de travail :", DOSSIER)

### 13.2 Chargement des fichiers enregistrés

Le modèle, le standardiseur et la médiane proviennent du notebook d’entraînement. **Aucun nouvel apprentissage n’est effectué ici.**

In [ ]:
modele = joblib.load(DOSSIER / "modele_ONCFM.pkl")
scaler = joblib.load(DOSSIER / "scaler_ONCFM.pkl")
mediane = joblib.load(DOSSIER / "mediane_ONCFM.pkl")
print("Modèle ONCFM chargé avec succès !")
print("Classes du modèle :", modele.classes_)
print("Médiane enregistrée :", mediane)

### 13.3 Importation et contrôle du nouveau CSV

Le fichier d’entrée peut contenir une colonne `id`. Les six mesures obligatoires doivent être présentes.

In [ ]:
fichier_entree = DOSSIER / "billets_production.csv"
billets = pd.read_csv(fichier_entree)
manquantes = [c for c in VARIABLES if c not in billets.columns]
if manquantes:
    raise ValueError(f"Colonnes requises absentes : {manquantes}")
if billets.empty:
    raise ValueError("Le CSV ne contient aucun billet.")
print("Nombre de billets :", len(billets))
display(billets.head(10))

### 13.4 Traitement des valeurs manquantes et standardisation

Seule `margin_low` est imputée avec la médiane enregistrée pendant l’entraînement. Le standardiseur chargé est appliqué sans être réajusté.

In [ ]:
X_nouveaux = billets[VARIABLES].copy()
for colonne in VARIABLES:
    X_nouveaux[colonne] = pd.to_numeric(X_nouveaux[colonne], errors="raise")
autres_manquantes = X_nouveaux.drop(columns=["margin_low"]).isna().sum()
if autres_manquantes.any():
    raise ValueError(f"Valeurs manquantes non prises en charge : {autres_manquantes[autres_manquantes > 0].to_dict()}")
X_nouveaux["margin_low"] = X_nouveaux["margin_low"].fillna(float(mediane))
X_nouveaux_scaled = scaler.transform(X_nouveaux[VARIABLES])
print("Dimensions des données standardisées :", X_nouveaux_scaled.shape)

### 13.5 Prédictions et scores de confiance

Le score de confiance est la probabilité attribuée par le modèle à **la classe prédite**. Il ne garantit pas que la prédiction est correcte.

In [ ]:
def libelle_classe(valeur):
    if isinstance(valeur, (bool, np.bool_)):
        return "Vrai billet" if valeur else "Faux billet"
    if isinstance(valeur, (int, np.integer, float, np.floating)) and valeur in (0, 1):
        return "Vrai billet" if valeur == 1 else "Faux billet"
    texte = str(valeur).strip().lower()
    if texte in ("true", "vrai", "vrai billet", "genuine", "1"):
        return "Vrai billet"
    if texte in ("false", "faux", "faux billet", "counterfeit", "0"):
        return "Faux billet"
    raise ValueError(f"Classe inattendue : {valeur!r}")

predictions = modele.predict(X_nouveaux_scaled)
probabilites = modele.predict_proba(X_nouveaux_scaled)
indices = [list(modele.classes_).index(p) for p in predictions]
confiances = [round(float(probabilites[i, indice]) * 100, 2)
              for i, indice in enumerate(indices)]
resultats = billets.copy()
resultats["Prediction"] = [libelle_classe(p) for p in predictions]
resultats["Confiance (%)"] = confiances
display(resultats)

### 13.6 Exportation des résultats

In [ ]:
fichier_sortie = DOSSIER / "resultats_ONCFM.csv"
resultats.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
print("Résultats enregistrés avec succès :", fichier_sortie)

## 14. Script Python intégré au notebook

La cellule suivante contient le **script complet** et permet de le créer automatiquement dans le même dossier. Elle n’exécute pas les prédictions une deuxième fois. Le script généré peut ensuite être lancé depuis PowerShell.

**Attention :** si un fichier `.py` du même nom existe déjà, la cellule le remplace. Sauvegarder l’ancien fichier si nécessaire.


In [ ]:
from pathlib import Path

# Script Python complet intégré à ce livrable unique.
# Exécuter cette cellule pour créer le fichier .py utilisable dans PowerShell.
SCRIPT_APPLICATION = '# -*- coding: utf-8 -*-\n"""ONCFM - Application de detection des faux billets.\nPlacer ce script avec les trois fichiers .pkl et billets_production.csv.\n"""\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nDOSSIER = Path(__file__).resolve().parent\nVARIABLES = ["diagonal", "height_left", "height_right", "margin_low", "margin_up", "length"]\n\n\ndef libelle_classe(valeur):\n    """Les classes d\'entrainement sont is_genuine (True = vrai)."""\n    if isinstance(valeur, (bool, np.bool_)):\n        return "Vrai billet" if valeur else "Faux billet"\n    if isinstance(valeur, (int, np.integer, float, np.floating)) and valeur in (0, 1):\n        return "Vrai billet" if valeur == 1 else "Faux billet"\n    texte = str(valeur).strip().lower()\n    if texte in ("true", "vrai", "vrai billet", "genuine", "1"):\n        return "Vrai billet"\n    if texte in ("false", "faux", "faux billet", "counterfeit", "0"):\n        return "Faux billet"\n    raise ValueError(f"Classe inattendue dans le modele : {valeur!r}")\n\n\ndef main():\n    modele = joblib.load(DOSSIER / "modele_ONCFM.pkl")\n    scaler = joblib.load(DOSSIER / "scaler_ONCFM.pkl")\n    mediane = joblib.load(DOSSIER / "mediane_ONCFM.pkl")\n    print("Modele ONCFM charge avec succes !")\n\n    fichier_entree = DOSSIER / "billets_production.csv"\n    billets = pd.read_csv(fichier_entree)\n    manquantes = [c for c in VARIABLES if c not in billets.columns]\n    if manquantes:\n        raise ValueError(f"Colonnes requises absentes du CSV : {manquantes}")\n    if billets.empty:\n        raise ValueError("Le fichier CSV ne contient aucun billet.")\n\n    X = billets[VARIABLES].copy()\n    for colonne in VARIABLES:\n        X[colonne] = pd.to_numeric(X[colonne], errors="raise")\n    # L\'imputation apprise pendant l\'entrainement concerne margin_low.\n    # Les autres colonnes ne doivent pas contenir de valeurs manquantes.\n    autres_manquantes = X.drop(columns=["margin_low"]).isna().sum()\n    if autres_manquantes.any():\n        raise ValueError(f"Valeurs manquantes non prises en charge : {autres_manquantes[autres_manquantes > 0].to_dict()}")\n    X["margin_low"] = X["margin_low"].fillna(float(mediane))\n\n    # Conserver l\'ordre exact des six variables utilise a l\'entrainement.\n    X_scaled = scaler.transform(X[VARIABLES])\n    predictions = modele.predict(X_scaled)\n    probabilites = modele.predict_proba(X_scaled)\n    indices = [list(modele.classes_).index(p) for p in predictions]\n    confiances = [round(float(probabilites[i, indice]) * 100, 2)\n                  for i, indice in enumerate(indices)]\n\n    resultats = billets.copy()\n    resultats["Prediction"] = [libelle_classe(p) for p in predictions]\n    resultats["Confiance (%)"] = confiances\n    print("\\nResultats de l\'analyse ONCFM :")\n    print(resultats.to_string(index=False))\n    sortie = DOSSIER / "resultats_ONCFM.csv"\n    resultats.to_csv(sortie, index=False, encoding="utf-8-sig")\n    print(f"\\nResultats enregistres avec succes : {sortie}")\n    return resultats\n\n\nif __name__ == "__main__":\n    main()\n'

chemin_script = Path.cwd() / "AGNEROH_Roger_2_Application_ONCFM_092026.py"
chemin_script.write_text(SCRIPT_APPLICATION, encoding="utf-8")
print(f"Script Python créé : {chemin_script}")


## 15. Bilan et utilisation en soutenance

Ce notebook constitue **un seul livrable applicatif** : explications, importation, prétraitement, prédiction, exportation et génération du script Python.

1. Placer ce notebook avec `modele_ONCFM.pkl`, `scaler_ONCFM.pkl`, `mediane_ONCFM.pkl` et `billets_production.csv`.
2. Exécuter les cellules dans l’ordre, y compris la cellule de génération du script.
3. Pour la démonstration PowerShell, lancer `python ".\AGNEROH_Roger_2_Application_ONCFM_092026.py"` depuis le dossier du projet.
4. Vérifier `resultats_ONCFM.csv` et expliquer les prédictions, sans les confondre avec des étiquettes réelles vérifiées.
